# 01 EDA：ESCI（英語・US）を自分の目で見る

データ：`../data/examples_us.parquet`（クエリ × 商品 × ラベル）、`../data/products_us.parquet`（商品の情報）

ルール：コードは自分で書く。分からない関数は調べる。答えを見る前に、まず予想を1行書く。

In [1]:
import pandas as pd

## 1. ファイルを開く
- 列は何がある？ 行は何件？（`shape`・`columns`・`dtypes`）
- 予想：

In [16]:
expample_data = pd.read_parquet('../data/examples_us.parquet')
df = pd.DataFrame(expample_data)
df.head(20)


,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,16,!awnmower tires without rims,1,B075SCHMPY,us,I,1,1,train
1,17,!awnmower tires without rims,1,B08L3B9B9P,us,E,1,1,train
2,18,!awnmower tires without rims,1,B082K7V2GZ,us,I,1,1,train
3,19,!awnmower tires without rims,1,B07P4CF3DP,us,S,1,1,train
4,20,!awnmower tires without rims,1,B07C1WZG12,us,E,1,1,train
5,21,!awnmower tires without rims,1,B077QMNXTS,us,E,1,1,train
6,22,!awnmower tires without rims,1,B01M3SJ699,us,C,1,1,train
7,23,!awnmower tires without rims,1,B06XX6BM2R,us,E,1,1,train
8,24,!awnmower tires without rims,1,B01DBGLLUC,us,S,1,1,train
9,25,!awnmower tires without rims,1,B00N16T5D8,us,I,1,1,train


In [25]:
print(df.shape)
print(df.columns)
print(df.dtypes)

(601354, 9)
Index(['example_id', 'query', 'query_id', 'product_id', 'product_locale',
       'esci_label', 'small_version', 'large_version', 'split'],
      dtype='str')
example_id        int64
query               str
query_id          int64
product_id          str
product_locale      str
esci_label          str
small_version     int64
large_version     int64
split               str
dtype: object


In [26]:
product_data = pd.read_parquet('../data/products_us.parquet')
df_products = pd.DataFrame(product_data)
df_products.head(20)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B0188A3QRM,"Amazon Basics Woodcased #2 Pencils, Unsharpene...",NaN,144 woodcase #2 HB pencils made from high-qual...,Amazon Basics,Yellow,us
1,B075VXJ9VG,"BAZIC Pencil #2 HB Pencils, Latex Free Eraser,...",<p><strong>BACK TO BAZIC</strong></p><p>Our go...,&#11088; UN-SHARPENED #2 PREMIUM PENCILS. Each...,BAZIC Products,12-count,us
2,B07G7F6JZ6,Emraw Pre Sharpened Round Primary Size No 2 Ju...,<p><b>Emraw Pre-Sharpened #2 HB Wood Pencils -...,✓ PACK OF 8 NUMBER 2 PRESHARPENED BEGINNERS PE...,Emraw,Yellow,us
3,B07MGKC3DD,"BIC Evolution Cased Pencil, #2 Lead, Gray Barr...",NaN,Premium #2 HB lead pencils with break-resistan...,Design House,Gray,us
4,B07NH2D2CP,Maped Essentials Triangular Graphite #2 Pencil...,NaN,Classic Yellow #2 Graphite Pencils with a mode...,Maped Helix USA,Yellow,us
5,B07Z78H162,"BIC Xtra Fun Cased Pencil, 2 Lead, Assorted Ba...",NaN,Kid-friendly pencils feature bright two-toned ...,BIC,NaN,us
6,B084Q4R7SG,Holographic Pencils with Erasers Metallic Asso...,<br>36 Pieces holographic pencils <br>Enough q...,5 Colors: there are 36 pieces holographic penc...,Outus,NaN,us
7,B087J9MBXJ,YOTINO Pre-sharpened Wood Cased #2 HB Pencils ...,</br>YOTINO Pre-sharpened #2 HB Pencils Pack o...,Convenient Bulk Pack - Box of 100 count #2 HB ...,N2,Yellow,us
8,B0891R2YFN,"Wood-Cased #2 HB Pencils, Shuttle Art 600 Pack...",NaN,600 COUNT VALUE PACK: Shuttle Art 600 Pack #2 ...,Shuttle Art,Yellow,us
9,B08KXRY4DG,"AHXML#2 HB Wood Cased Graphite Pencils, Pre-Sh...","<b>AHXML#2 HB Wood Cased Graphite Pencils, Pac...","#2 HB yellow, wood-cased pencils:Box of 48 cou...",AHXML,NaN,us


In [27]:
df_products.shape
print(df_products.columns)
print(df_products.dtypes)

Index(['product_id', 'product_title', 'product_description',
       'product_bullet_point', 'product_brand', 'product_color',
       'product_locale'],
      dtype='str')
product_id              str
product_title           str
product_description     str
product_bullet_point    str
product_brand           str
product_color           str
product_locale          str
dtype: object


### 📝 1 のメモ（自分の気づき・2026-10-23）
- examples：1つのクエリに対して商品がたくさん並び、それぞれが E/S/C/I のどれかと商品 ID が付いている → 「実際の検索結果の採点表」っぽい
- products：ID だけでなく商品の説明がある。ただし**説明があるものとない（NA）ものがある**
- 疑問①：なぜ NA になっている？（元々書かれていない？取れなかった？）
- 疑問②：検索で一番情報が多いはずの説明文がないと、その商品は不利になる？
- 行数：examples 約60万行、products 約48万行。列は 9 個
- 分からないこと（後で）：元のデータから examples と products に分ける方法・parquet の中身

## 2. 1行ずつ読む
- クエリ・商品タイトル・ラベルを5件ずつ、声に出して読む
- 2つのファイルは何の列でつながる？
- 予想：Product ID, Product locale

これ今上のColの処理で同じのとか、目で見て差しかめた
けど、これgrepとかで同じの探すとかできると思った　あるいはpdであるならしりたい
次元が１０２４とかのやつもあるから

これを見て思ったこと
Exampleにかんしては、1 vs 15 productの対になってる。それぞれのESCIかテストかで分かれてるけど、んぜここで分かれてるの？

Smallversion vs largeversionがわからない・Queryのサイズの違い？

In [28]:
df['query'].unique()[:20]

<ArrowStringArray>
[                           '!awnmower tires without rims',
                            '!qscreen fence without holes',
                 '# 10 self-seal envelopes without window',
                               '# 2 pencils not sharpened',
                                              '# mom life',
   '#1 best and not expensive bath back brush cream color',
   '#1 black natural hair dye without ammonia or peroxide',
      '#1 rated resveratrol supplement without tea leaves',
             '#1 selling shoes for men without shoeleases',
        '#1 small corded treadmill without remote control',
                     '#10 envelopes without security tint',
            '#10 standard no tint no window not self seal',
                      '#10 window envelopes not self seal',
                    '#10 window envelopes without plastic',
                           '#11 mrs. kormel is not normal',
 '#12 black boys chain necklace without baseball stitches',
                     

In [34]:
onequery_df = df[df['query']=='#1 selling shoes for men without shoeleases']
onequery_df.head(20)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
270,482,#1 selling shoes for men without shoeleases,14,B07FBGMW49,us,E,1,1,train
271,483,#1 selling shoes for men without shoeleases,14,B08FY583NX,us,E,1,1,train
272,484,#1 selling shoes for men without shoeleases,14,B075686DKD,us,S,1,1,train
273,485,#1 selling shoes for men without shoeleases,14,B07932FK2J,us,S,1,1,train
274,486,#1 selling shoes for men without shoeleases,14,B079Y2MP8C,us,S,1,1,train
275,487,#1 selling shoes for men without shoeleases,14,B07B3VQ8DY,us,I,1,1,train
276,488,#1 selling shoes for men without shoeleases,14,B07C2Y4JBG,us,S,1,1,train
277,489,#1 selling shoes for men without shoeleases,14,B07FKVC82C,us,E,1,1,train
278,490,#1 selling shoes for men without shoeleases,14,B074F8699Z,us,E,1,1,train
279,491,#1 selling shoes for men without shoeleases,14,B07G7TDPRL,us,S,1,1,train


In [35]:
pd.merge(onequery_df, df_products, on=['product_id', 'product_locale'], how='left')

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,482,#1 selling shoes for men without shoeleases,14,B07FBGMW49,us,E,1,1,train,"Skechers mens Cessnock Food Service Shoe, Blac...",NaN,Breathable mesh\nBungee lacing\nSlip resistant...,Skechers,Black
1,483,#1 selling shoes for men without shoeleases,14,B08FY583NX,us,E,1,1,train,JSLEAP Mens Running Shoes Walking Non Slip Spo...,<b>Fashion Unique Design Perfect Choice To Pai...,★<b>Fashion Knitted Mesh Upper</b>: Ultra-ligh...,JSLEAP,5dark Green
2,484,#1 selling shoes for men without shoeleases,14,B075686DKD,us,S,1,1,train,Skechers Men's Relaxed Fit-Elent-Mosen Boat Sh...,NaN,Air Cooled Memory Foam\nRelaxed Fit\nAir Coole...,Skechers,Black
3,485,#1 selling shoes for men without shoeleases,14,B07932FK2J,us,S,1,1,train,"Hey Dude Men's Wally Sox Ash, Size 10",<ul> <li>The Hey Dude™ Wally Sox will be an in...,30-DAY MONEY BACK GUARANTEE: If you are not s...,Hey Dude,Ash
4,486,#1 selling shoes for men without shoeleases,14,B079Y2MP8C,us,S,1,1,train,Xero Shoes Men's Prio Cross Training Shoe - Li...,"""The Prio is the best minimalist shoe I've eve...","COMFORTABLE, FOR-EVERYTHING SHOE - Whether you...",Xero Shoes,Black
5,487,#1 selling shoes for men without shoeleases,14,B07B3VQ8DY,us,I,1,1,train,New Balance Men's 608 V5 Casual Comfort Cross ...,NaN,Leather upper\nABZORB midsole cushioning aids ...,New Balance,White/Navy
6,488,#1 selling shoes for men without shoeleases,14,B07C2Y4JBG,us,S,1,1,train,New Balance Women's FuelCore Nergize V1 Sneake...,NaN,Midsole Cushioning: These New Balance sneakers...,New Balance,Black/Magnet
7,489,#1 selling shoes for men without shoeleases,14,B07FKVC82C,us,E,1,1,train,Skechers Men's GO Walk Evolution Ultra-Impecca...,NaN,The leaders in walking technology present the ...,Skechers,Khaki
8,490,#1 selling shoes for men without shoeleases,14,B074F8699Z,us,E,1,1,train,"Crocs Men's LiteRide Pacer Sneaker, Black/Whit...",NaN,SCIENCE AND MAGIC: Take a ride in a superbly c...,Crocs,Black/White
9,491,#1 selling shoes for men without shoeleases,14,B07G7TDPRL,us,S,1,1,train,Under Armour mens Charged Assert 8 Running Sho...,NaN,NEUTRAL: For runners who need a balance of fle...,Under Armour,Black (002)/Black


### 📝 2 のメモ（自分の気づき・2026-10-23）
- つなぐ列：`product_id`（`how='left'` で、そのクエリの判定を全部残して商品情報を足す）。本当は `['product_id', 'product_locale']` の方がきれい
- E（ぴったり）は 20件中15件 → 悪くない
- 判断の仕方：クエリに対して、商品タイトルと説明で判断した。箇条書きは表示が小さくて読めなかった
- I（関係ない）：「#1」のように用途が絞られていないクエリなので、スポーツもファッションも混ざる → 検索としては妥当
- 自分なら「靴紐がない、いいもの」が欲しい。メレル（自分で履いている）は運動・歩きやすさ・ファッションを兼ねていて、それに近いものを想像した
- → 「紐なし」は商品側では "slip-on" と書かれることが多いはず（Amazon の否定書き換え論文と同じ「言葉のずれ」）

In [36]:
pd.set_option('display.max_colwidth', None)


## 3. 数える
- ラベル（E/S/C/I）ごとの件数・割合　Sが意外と多い　Supplementaryは欲しいものがない時に見るといやかも。Iもおおいし
- クエリは何種類？ 1クエリあたり商品は何件？　#これみてて、Queryが端的に靴！みたいな感じで品の名称がないけど見落としてるだけ？　
- 商品に値段・星・レビューはある？　なかった
- 予想：さっき紐なしのやつで１５、２０とかだったからこれくらいかな

見てみて
Aipodとか人気のやつはヒットした検索多いね
このクエリは否定とかは何もないからそれにヒットする奴が多いの？

In [44]:
print(df['esci_label'].value_counts())
print(df['esci_label'].value_counts(normalize=True))
print()
print(df['query'].unique().shape)
print(df['query'].value_counts())

esci_label
E    261527
S    211191
I    101447
C     27189
Name: count, dtype: int64
esci_label
E    0.434897
S    0.351192
I    0.168698
C    0.045213
Name: proportion, dtype: float64

(29844,)
query
airpods                           188
printer                           136
face mask                         109
apple watch series 3               98
kindle                             95
                                 ... 
woman coat                          8
womens tall pants                   8
zac brown band cd welcome home      8
zagg 44mm                           8
가마솥                                 8
Name: count, Length: 29844, dtype: int64


In [45]:
df.tail(20)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
601334,2614591,香奈儿,130378,B006IB5T4W,us,I,1,1,test
601335,2614592,香奈儿,130378,B0010POWEE,us,I,1,1,test
601336,2614593,香奈儿,130378,B081X6DRRT,us,E,1,1,test
601337,2618538,가마솥,130537,B0984JBPYH,us,C,1,1,train
601338,2618539,가마솥,130537,B07Y5L6XYM,us,E,1,1,train
601339,2618540,가마솥,130537,B07QDYP2NZ,us,S,1,1,train
601340,2618541,가마솥,130537,B07GKZZMSF,us,C,1,1,train
601341,2618542,가마솥,130537,B0793N5STW,us,I,1,1,train
601342,2618543,가마솥,130537,B000N4UX4Q,us,E,1,1,train
601343,2618544,가마솥,130537,B00063RXQK,us,E,1,1,train


### 📝 3 のメモ（自分の気づき・2026-10-23）
- S は Supplementary ではなく **Substitute（代わりになる物）**。欲しい物そのものがないときに見るもの → 割合（E 43%・S 35%・I 17%・C 5%）も納得
- クエリは 29,844 種類。「靴」のように短いクエリが多い（本物の検索窓の言葉なので普通）
- ESCI の行数＝**人が採点した候補の数**。「検索でヒットした数」ではない。8行のクエリなら、8商品が E/S/C/I のどれかに振り分けられている（合計8）
- 人気のクエリ（airpods など）は採点された候補が多い → 人気の目安になる？ ただし採点の選び方（Amazon の設計）にも左右されるので、そのまま人気度とは言えない
- locale が US なのに韓国語のクエリがある（tail で確認）。自分がドイツで日本語で検索するのと同じ。こういうクエリは件数が少なめ → 需要が少ない？ ただしこのデータだけでは言い切れない
- → 多言語のクエリは「もう1言語」をやるときの材料になる

## 4. 否定・条件つきのクエリを抜く
- `without` / `no` / `free` / `not` / `except` などで何件？　6000くらいかな。けど、何で日本語のやつが2300ってわかったの？　基本は靴　サイズ　色　ブランド　防水機能みたいな感じかな
- 20件読んで、本物の条件（例：keyboard without numpad）と、ただの商品名（例：No Fear）を分ける
- 予想：これにRufusに聞いたらどう理解されるのだろう？Without xxってのは文を聞いていれば逆なんだなってので頭に入るけど、Attentionがここを重要って思ってくれるのかな？これってAttentionの問題？

result
思ったより少ない　さっきの日本語のが一万件の中で２，３００自分で確認できてないからあれだけど
3万だと３倍にはならなかったね

言語的にもNoXXとかじゃなくてグルテンフリーとかNegativeがそもそもわかりにくい感じの単語が存在してるのかも？
今回は否定だったけど、制限　500mlいないとかもあればふえるのかな

今4番みて少し変更した　スペース入れる。入れないでをやったら少し減った

case=Falseにしたら今度は増えた　Without appleみたいな感じで否定から入るのもあるからね

In [49]:
q = df['query'].drop_duplicates().reset_index(drop=True)
print(q.shape)

negative_q = q[q.str.contains('without')]
print(negative_q.shape)

negative_q = q[q.str.contains('without | not')]
print(negative_q.shape)

negative_q = q[q.str.contains('without | not | no')]
print(negative_q.shape)

negative_q = q[q.str.contains('without | not | no | non', case=False) ]
print(negative_q.shape)

negative_q = q[q.str.contains('without | not ')]
print(negative_q.shape)

negative_q = q[q.str.contains('without | not | no ')]
print(negative_q.shape)

negative_q = q[q.str.contains('without | not | no | non ', case=False)]
print(negative_q.shape)

(29844,)
(1726,)
(2466,)
(2614,)
(2448,)
(2386,)
(2425,)
(2441,)


### 📝 4 のメモ（自分の気づき・2026-10-23）
- 予想 6,000 → 実際は約 2,400〜2,600（約8%）。思ったより少ない。日本語版（1万中 約2,300・ただし本のタイトルなども混ざる）の3倍にはならなかった
- `without` だけで 1,726。`not`・`no` を足すと 2,466 → 2,614
- 空白を入れる／入れないで数が変わる。`case=False` で増える（"Without apple" のように文の頭で大文字の否定がある）
- ⚠️ 正規表現の注意：`'without | not'` は「"without " または " not"」。`|` の前後の空白も文字として数えられる → 文の最後の without や、文の頭の not を取りこぼす。逆に ` no` は " notebook" も拾う。**数より「何が入って何が漏れたか」を目で見る**のが大事
- 言葉の気づき：「gluten free」のように、no/not を使わない否定がある（free＝〜なし）。否定の単語だけで探すと漏れる
- 広げるなら：数量の条件（500ml 以内）も入れると増えそう → 計画どおり「否定 → 数・量」の順
- 疑問：Rufus は "without X" をどう理解する？ Attention が without を重要と見てくれるか？ → 段6「attention を見る」でそのまま確かめられる問い
- まだ：20件を読んで「本物の条件」と「ただの商品名」に分ける（次回）

In [51]:
#仕分け
#否定の取り出しコマンド
negative_q_key = r'\b(without|not|no|non|free|except|exclu\w*)\b'

#pick negative random 20 queries
df_negative = df[df['query'].str.contains(negative_q_key, case=False)].sample(n=20, random_state=42)
df_negative

/var/folders/k_/xfq6fm9x7419_ttdxwb035v80000gn/T/ipykernel_42385/3607178433.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_negative = df[df['query'].str.contains(negative_q_key, case=False)].sample(n=20, random_state=42)


,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
187071,650717,delta pop up drain without overflow,32292,B002CP3030,us,E,1,1,train
244109,882898,gaming headphones without microphone,44244,B09H2G9WWC,us,I,1,1,train
17280,25509,1 tsp of grenetine without sweetener,900,B01JI1EV4C,us,S,1,1,train
432585,1610197,polyvisol without iron,82027,B07HYXHZVW,us,S,1,1,test
269622,974032,hang clothes without closet,48998,B07D1F3DSR,us,E,1,1,test
69335,175955,alarm clock without radio,7751,B08X4X5B62,us,E,1,1,test
517627,1946036,swanson multivitamin without iron,99632,B00068U1QE,us,S,1,1,train
14943,22173,1 inch mini curling iron with case not straighteners,779,B089GY2MQZ,us,S,1,1,train
596694,2243678,yorktend not a day over fabulous wine tumbler,115094,B081N832JY,us,S,1,1,test
11370,17374,1 1/2 pop up drain without overflow,621,B07V5VTZSS,us,E,1,1,train


650717否定
882898否定
25509否定
1610197否定
974032否定
175955否定
1946036否定
22173否定
2243678多分否定じゃない商品の名前かな
17374否定
943563否定
2106568否定
745316否定+オフ　このオフも否定というか、ネガティブなところとかこれなしでみたいなところで使ったりもするのかなと思った。
1835565否定
1799250否定の形で使われているけれども、何かを外して欲しい。否定じゃなくて、何かの文章の途中っぽく見えるから違うと思った。
872064否定
2144107 Tシャツのロゴかな
230092否定
1629606否定
169038否定
こうしてみると、ウィズアウトがすごい多いなと思った。予想通りだね。アレルギーとかのグルテンフリーとかも単語とはあるけど、一般ユーザが多いって考えたときに、これが1番あるってやつなのかなとなった時にウィザードさえできていれば、割と楽であってもエージェントであってもしっかり言うこと聞いてくれるんじゃないかなと思った。例えばさっきのです。コード話した時もこれは書かないでとかこれは違うみたいなことを言っただけなんと理解してくれと言うから、もう解決されているのではと思ってしまうけど、これは間違いなのかな？まだこの否定がちゃんと理解されていないのか否かっていうところの境界線がまだちょっとわからないかも。


### 📝 4 の仕分け結果（2026-10-24・`random_state=42` の20件）
- ✅ 本物の条件 16・❌ 名前や言葉の一部 4（約8割が本物）→ 約2,500クエリなら本物は約2,000件と見積もれる
- ❌ の例：yorktend「not a day over fabulous」（タンブラーの文字）、do not disturb sign（看板）、watch the game not your phone（Tシャツ）、greg olsen「Worlds Without End」（絵のタイトル）
- 🤔 寄り：hang clothes without closet（物の性質ではなく状況）
- `without` が圧倒的に多い。一般の人が一番使う否定
- コード：`\b`（単語の境目）で no→nokia のような誤ヒットを防ぐ。`exclu\w*` は exclude/excluding/exclusive を拾う（exclusive は ❌ で外す）。`(?:...)` で警告が消える


## 5. 商品の説明に条件は書いてある？
- 4 で選んだクエリ3つについて、E（ぴったり）の商品と I（関係ない）の商品の説明文を読む
- 条件（numpad がない等）は、タイトル・説明・箇条書きのどこに書いてある？ 書いてない？

69335	175955	alarm clock without radio	7751	B08X4X5B62	us	E	1	1	test

gaming headphones without microphone（I）
244109	882898	gaming headphones without microphone	44244	B09H2G9WWC	us	I	1	1	train

artificial roses without stems（I）
83073	230092	artificial roses without stems	10568	B07WFJW17R	us	I	1	1	train

electric heating pad without auto shut off（S）

In [ ]:
query_id_list = [7751,44244,10568]
df_filtered = df[df['query_id'].isin(query_id_list)]
df_filtered

#merge with product data
df_merged = pd.merge(df_filtered, df_products, on = ['product_id', 'product_locale'], how='left')
df_merged[df_merged['query_id'] == 7751].sort_values(by='esci_label')

# df_merged['product_description'].str.contains('radio', case=False, na=False)



,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,175953,alarm clock without radio,7751,B09CDHJ4PY,us,E,1,1,test,"RunningByte Digital Alarm Clock Radio- AM/FM Radio with preset, Large LED Display with Dimmer, Dual Alarm, Adjustable Alarm Volume, Snooze, Sleep Timer, Easy to Operate Clock Radios for Bedroom","<b>Decent Alarm Clock with Large Clear Display</b> <br> This clock radio equipped with a large clear red LED screen, the time can be easily read anywhere in the room, even without your glasses. <br>Decent look, perfect size, simple and beautiful design. It can even be used as an additional decoration. Suitable for living room, kitchen, office, bedroom, etc.<br> <br> <b>Thoughtful Dual-alarm Design</b> <br>You can set two separate wake-up alarms for you and your partner on the same day. <br> <br> <b>Enjoy FM/ AM Radios</b> <br>Turn on the radio, lean on the couch and enjoy your favorite channels, music, or news. 40 stations can be preset (20 AM+20 FM). Built-in high-quality loudspeaker, you can adjust the radio volume from level 0 to level 16. <br> <br> <b>Programmable Sleep Timer</b> <br>Fall asleep in your favorite radio, enjoy a soothing and pleasant bedtime before sleeping. 10-90 minutes optional. <br> <br> <b>Repeating Snooze</b> <br>Enjoy a few more sweet dreams thanks to the alarm clock’s handy snooze function. Sometimes that a little bit of extra sleep is all it takes to wake up feeling refreshed. <br> <br> <b>Battery Backup</b> <br>The function of 2 AAA backup batteries (not included) is to save the time and alarm settings in case of main power outage. During the battery backup, the screen will go blank, but the clock will display correct time once power returns. <br> <br> <b>What’s in the package? </b><br><br> 1. Alarm Clock Radio *1 <br> 2. User Manual *1 <br> 3. Warranty Card *1 <br> <br> <b>Your Purchase is RISK-FREE! </b> <br>If you are not happy with our product, please contact us. <br>We can provide 90-day money back and 365-day free replacement. <br><b>ADD TO CART NOW! GET IT WITH CONFIDENCE! </b>","Elegant Digital Alarm Clock radio with Dimmer for Bedroom- 4.9 Inches large LED screen with Red number display, eyes-friendly and easy to read. According to your preference, the screen has 4 levels of screen brightness that can be easily set, no need to worry about too high brightness will affect your sleep or too dark to read time.\nSimple Dual Alarm Clock with Easy Snooze- Two alarms at different time can be set, Wake up to buzzer or radio should your preference. SNOOZE (for 9 minutes)-The largest button on the top is easily accessible even in semi-sleep status.\nDigital AM / FM Radio with Sleep Timer- Easy digital tuning in AM FM radio channel with 0-15 level volume adjustable. The Sleep timer function lets you fall asleep to the radio by preset the radio to play for a certain amount time (10-90 minutes optional).\nPreset Radio Stations- Support preset up to 40 stations (20 AM+20 FM), Preset your favorite stations and enjoy wonderful programs easily.\nAC Powered with Battery Backup- Plug into any wall outlet(110V) to power the clock radio. The function of the Battery Backup is to save the time and alarm you have set. During the battery backup, the screen will go blank, but the clock will display correct time once the AC power returns.",RunningByte,NaN
1,175954,alarm clock without radio,7751,B098LK6GQJ,us,E,1,1,test,"Sunrise Alarm Clock, Wake Up Light with Sunrise/Sunset Simulation, Dual Alarms with FM Radio, 7 Nature Sounds & Snooze, 7 Colors Night Light, Bedroom Digital Alarm Clock for Heavy Sleepers Adults Kids","<p><b> Who Can Benefit From Sunrise Alarm Clocks? </b><br> People having a hard time waking up in the dark or during seasons with late morning sunshine. <br> For working individuals who need to wake up at a specific time. <br> For people suffering from sleep and behav

In [65]:
df_q = df_merged[df_merged['query_id'] == 7751].copy()
df_q['in_title'] = df_q['product_title'].str.contains('radio', case=False, na=False)
df_q['in_desc'] = df_q['product_description'].str.contains('radio', case=False, na=False)
df_q[['esci_label', 'in_title', 'in_desc', 'product_title']].sort_values('esci_label')

,esci_label,in_title,in_desc,product_title
0,E,True,True,"RunningByte Digital Alarm Clock Radio- AM/FM Radio with preset, Large LED Display with Dimmer, Dual Alarm, Adjustable Alarm Volume, Snooze, Sleep Timer, Easy to Operate Clock Radios for Bedroom"
1,E,True,True,"Sunrise Alarm Clock, Wake Up Light with Sunrise/Sunset Simulation, Dual Alarms with FM Radio, 7 Nature Sounds & Snooze, 7 Colors Night Light, Bedroom Digital Alarm Clock for Heavy Sleepers Adults Kids"
2,E,False,False,"SHARP Digital Alarm Clock with Keyboard Style Controls, Battery Back-up, Easy to Use with Simple Operation, Black Case with Red LED Display"
3,E,False,True,"DreamCaster by Sharp Sound Machine Alarm Clock with Bluetooth Speaker, 6 High Fidelity Sleep Machine Soundtracks – White Noise Machine for Baby, Adults, Home and Office – Blue LED"
4,E,False,False,"Sharp Digital Alarm Clock – Black Case with Red LEDs - Ascending Alarm Grows Increasing Louder, Gentle Wake Up Experience, Dual Alarm - Battery Back-up, Easy to Use with Simple Operation"
6,E,True,True,"HAPTIME Digital Alarm Clock Radio: 6.2” Large LED Display with 4 Brightness Dimmer, Dual Alarms, Snooze, 12/24H, FM Radio with Sleep Timer, Blue Digits Clock for Home Bedside Bedroom"
7,E,False,False,"Small LED Digital Alarm Clock with Nightlight, Easy to Set, Full Range Brightness Dimmer, HI-Low Volume, USB Charging Port, Snooze, Outlet Power Clock for Senior Bedrooms, Bedside, Desk"
8,E,False,False,"Digital Alarm Clock, with Wooden Electronic LED Time Display, 3 Alarm Settings, Humidity & Temperature Detect, Wood Made Electric Clocks for Bedroom, Bedside… (Black)"
9,E,False,False,"USCCE Small LED Digital Alarm Clock with Snooze, Easy to Set, Full Range Brightness Dimmer, Adjustable Alarm Volume with 5 Alarm Sounds, USB Charger, 12/24Hr, Compact Clock for Bedrooms, Bedside, Desk"
10,E,False,False,"Sharp LED Digital Alarm Clock – Simple Operation - Easy to See Large Numbers, Built in Night Light, Loud Beep Alarm with Snooze, Bright Big Red Digit Display"


In [67]:
df_q = df_merged[df_merged['query_id'] == 44244].copy()
df_q['in_title'] = df_q['product_title'].str.contains('microphone', case=False, na=False)
df_q['in_desc'] = df_q['product_description'].str.contains('microphone', case=False, na=False)
df_q[['esci_label', 'in_title', 'in_desc', 'product_title']].sort_values('esci_label')

,esci_label,in_title,in_desc,product_title
58,E,False,False,"50MM Drivers Studio Headphones MAONO AU-MH601 Over Ear Stereo Monitor Closed Back Headphones for Music, DJ, Podcast (Black)"
59,E,True,True,"STOON Gaming Headset, for Xbox One PS4 Headset with Noise Canceling Microphone & LED Light, Over-Ear Gaming Headphones with Soft Memory Earmuffs for PC, Mac, Laptop, Nintendo Switch"
63,E,True,True,"SENZER SG500 Surround Sound Pro Gaming Headset with Noise Cancelling Microphone - Detachable Memory Foam Ear Pads - Portable Foldable Headphones for PC, PS4, PS5, Xbox One, Switch"
64,E,False,False,"Kikc PS4 Gaming Headset with Mic for Xbox One, PS5, PC, Mobile Phone and Notebook, Controllable Volume Gaming Headphones with Soft Earmuffs, Headphones for Kid"
67,E,False,False,"Logitech G435 Lightspeed and Bluetooth Wireless Gaming Headset - Lightweight Over-Ear Headphones, Built-in mics, 18h Battery, Compatible with Dolby Atmos, PC, PS4, PS5, Mobile - Black"
68,E,False,True,"YOWU RGB Cat Ear Headphone 3G Wireless Bluetooth 5.0 Foldable Gaming Headset with 7.1 Surround Sound, Built-in Mic & Customizable Lighting and Effect via APP, Type-C Charging Audio Cable-Black"
70,E,True,False,"FULAIM Boom Microphone Cable Compatible with Bose QuietComfort 35 (QC35) & Quiet Comfort 35 II (QC35 II) Headphones with Volume Control & Mute Switch for PC, Laptop, PS4 PS5 Xbox One Controller"
77,E,True,False,"HyperX Cloud Stinger Core – Wireless Gaming Headset, for PS4, PS5, PC, Lightweight, Durable Steel Sliders, Noise-Cancelling Microphone - White"
80,E,False,False,Sony MDR7506 Professional Large Diaphragm Headphone
82,E,False,False,"OneOdio Adapter-Free Over Ear Headphones for Studio Monitoring and Mixing, Sound Isolation, 90° Rotatable Housing with Top Protein Leather Earcups, 50mm Driver Unit, Wired Headsets with Mic (Pro-50)"


In [68]:
df_q = df_merged[df_merged['query_id'] == 10568].copy()
df_q['in_title'] = df_q['product_title'].str.contains('stem', case=False, na=False)
df_q['in_desc'] = df_q['product_description'].str.contains('stem', case=False, na=False)
df_q[['esci_label', 'in_title', 'in_desc', 'product_title']].sort_values('esci_label')

,esci_label,in_title,in_desc,product_title
19,E,True,True,"140 Pieces Artificial Stemless Rose Flower Heads, 3 Inches Artificial Foam Rose Head Without Stem for Weddings, Decor, DIY (Peach, Ivory )"
20,E,False,False,"Artificial Roses in 2 Pink Colors, 2-Inch Faux Flower Heads for Crafts (200 Pack)"
21,E,False,False,"Artificial Roses, 2-in Faux Flower Heads for Crafts, Decor (Black, White, 200 Pack)"
22,E,True,False,"Artificial White Rose Flower Heads for Crafts, Stemless Roses (2 in, 50 Pack)"
23,E,False,False,"Artificial Red Rose Silk Flower Head for Crafts, Wedding Decor (3.5 in, 36 Pack)"
25,E,False,True,"FTPRO 2 Artificial Flowers Bouquets of Roses, Hydrangea and Lilies. Realistic Fake Flower Arrangement is Suitable for Home Decoration of Bridal Bouquet, Living Room Dining Table(White)"
26,E,True,False,"Stemless Rose Flower Heads, Artificial Roses for Weddings and Crafts (3 in, Burgundy, 100 Pack)"
27,E,True,False,"Heave 25/50/100Pcs Real Touch Artificial Rose Head, Artificial 3D Foam Roses Flower Bulk Without Stem for DIY Wedding Bouquets Home Decoration Milk White 25Pcs"
28,E,False,False,"102 Pieces Fake Rose Flower Heads 1.4 Inch Mini Artificial Silk Rose Head for Crafts Wedding Home Decoration Party Marriage Car Shoes Hats Corsage Accessory Decor, 6 Colors"
32,E,True,False,"12 Pieces Halloween White Blood Rose Head Without Stem White Artificial Rose Flowers Bride Brooch Pin for Halloween Party Favor Home Atmosphere Decor DIY Hair Clips, White"


### 📝 5 のメモ（自分の気づき・2026-10-24）
**alarm clock without radio（7751・17件）**
- E 15件中3件がタイトルに radio（RunningByte・Sunrise・HAPTIME）＝ **約20%が without を無視した E**
- 最初に見た RunningByte「Clock Radio – AM/FM」は `split=test`（評価用）→ 否定を守るモデルほど点数が下がる
- ほぼ同じラジオ付き時計が E・S（Antomove）・I（DreamSky）とバラバラ＝ラベルの揺れ
- 条件（radio）は**タイトルに出ることが多い**。説明だけ True の物（DreamCaster）は「no radio」の可能性もあるので前後を読む

**gaming headphones without microphone（44244）**
- 違反がさっきより多い。ヘッドフォンは**マイク付きが当たり前** → マイクなしはニッチ
- mic / microphone の言い換え → 単語1つで探すと漏れる

**artificial roses without stems（10568）**
- E が多い。造花は**茎なし（stemless）もよくある**
- stem**less** は「茎なし」なのに `stem` で当たる → 単語一致だけでは意味は分からない

**ESCI の採点（論文 Reddy et al. 2022 で確認）**
- 1組を3人以上で採点 → 多数決。抜き取り確認で一致 91%（E か否かなら 96%以上）
- E の定義＝「条件を*全部*満たす」→ ラジオ付きの E はガイドライン上も間違い
- 採点者は amazon.com の商品詳細ページを見た。ラベルは全部人、候補の商品は検索システムが出したもの
- クエリはわざと難しい物を選んでいて、選び方の1つが「否定」
- train/test は同じ採点で、クエリ単位で 70/15/15 に分けただけ → train にも同じ揺れがある


## 6. 気づき（5行・Obsidian にも写す）
1. 否定クエリの約8%、そのうち約8割が本物の条件。`without` が一番多い
2. 人の多数決ラベルでも without は無視される（E 15件中3件が違反・test にも入っている）→ 今の物差しでは否定を守るモデルが損をする
3. **外す物がその種類の「当たり前」だと難しい**（ヘッドフォンのマイク）／当たり前でないと易しい（造花の茎）← 一番大事な仮説
4. 単語一致では意味は分からない：mic/microphone の言い換え、stemless に stem が当たる
5. LLM は読めば without が分かる。でもその**前の検索の段**で違反が混ざると直せない → ConstraintRAG が直すのはここ

**次にやること**：「without X」のクエリで、E の商品タイトルに X が入っている割合を全部数える（目で読まずに、否定を無視した E が何%あるか出す）
**置いておく問い**：Google 検索や他のサービスでも、否定で同じように崩れる？
